# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [1]:
!pip install langchain_community langchain_experimental langchain_openai

In [2]:
from typing import Optional

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [3]:
# %pip install faiss-cpu > /dev/null
# %pip install google-search-results > /dev/null
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [4]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [5]:
print(_)

True


In [6]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [7]:
from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain.chains import LLMChain
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI

todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}. Presentation must be in bulled points and prioritizing clarity. Maximum 20 words per line"
)
todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)
search = SerpAPIWrapper()

def search_design_inspiration(query: str) -> str:
    """
    Find real-world visual inspiration using SerpAPI.
    It performs three searches: general branding, logos, and color palettes.
    """
    search = SerpAPIWrapper()
    
    # Search 1: branding + design
    result1 = search.run(f"{query} branding visual identity design inspiration")
    
    # Search 2: logos 
    result2 = search.run(f"{query} logo design examples best")
    
    # Search 3: colors
    result3 = search.run(f"{query} color palette brand mood board")
    
    return (
        f"BRANDING & VISUAL IDENTITY\n{result1}\n\n"
        f"LOGO DESIGN EXAMPLES\n{result2}\n\n"
        f"COLOR PALETTE & MOOD\n{result3}"
    )

tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
    Tool(                                                         
        name="VisualSearch",                                      
        func=search_design_inspiration,                           
        description="Searches the internet for real visual inspiration: branding, logo examples, color palettes. Input: business concept or product name.",  # <-- NUEVO 2
    ),
]

prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""  # original
suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

/var/folders/dk/w5mlpx9x4t9fj30jk64759lh0000gn/T/ipykernel_15422/3958087460.py:9: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)


In [8]:
from langchain_openai import ChatOpenAI

llm = OpenAI(temperature=0, model="gpt-4o-mini")
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

/var/folders/dk/w5mlpx9x4t9fj30jk64759lh0000gn/T/ipykernel_15422/1595084072.py:6: LangChainDeprecationWarning: The class `ZeroShotAgent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use create_react_agent instead.
  agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)


### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [9]:
# OBJECTIVE = "Write a weather report for SF today"  #Original
OBJECTIVE = "Write a business plan for a vegan food truck"

In [10]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [11]:
baby_agi({"objective": OBJECTIVE})

/var/folders/dk/w5mlpx9x4t9fj30jk64759lh0000gn/T/ipykernel_15422/3867971396.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  baby_agi({"objective": OBJECTIVE})



*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: I need to create a todo list for writing a business plan for a vegan food truck.
Action: TODO
Action Input: Write a business plan for a vegan food truck
Observation: 

1. Research the vegan food truck industry and market trends.
2. Identify target audience and their preferences for vegan food.
3. Create a unique and appealing menu with a variety of vegan options.
4. Determine the startup costs and create a budget for the food truck.
5. Develop a marketing strategy to reach potential customers.
6. Secure necessary permits and licenses for operating a food truck.
7. Find a suitable location for the food truck to operate.
8. Source high-quality and sustainable ingredients for the menu.
9. Create a business plan outlining the mission, goals, and objectives of the food truck.
10. Develop a pricing strategy that is competitive and profitable.
11. Create a schedule for food preparation and service.
12. Hire and train staff for the food truck.
13. Purchase necessary equipment and supp

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: I need to gather information about existing vegan food trucks and their offerings.
Action: Search
Action Input: "vegan food truck competitors"
Observation: ['Vegan Van is probably the most successful. They have a major following. They rotate menus & do mock fast food - vegan Taco Bell week, vegan KFC ...', 'Taylor Herbert is one half of The Easy Vegan, the 2023 Great Food Truck Race winners. The two chefs have a combined 28 years of experience in ...', 'Expected 100-200 people no other food trucks besides a BBQ meat truck. We want to be able to provide both options and little competition.', 'Señoreata, a Cuban pop-up restaurant based in Los Angeles, made history when it won the 15th season of The Great Food Truck Race on the Food Network.', '00:00:15 Vegan food truck alternatives 00:00:52 Vegan Caribbean food ... Starting a Vegan Food Truck Business [ 15 Vegan Food Truck Ides ] Food ...', 'Review our extensive list of vegan food trucks in your city and discover the creative me

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: I need to outline the financial aspects of the vegan food truck business, including initial investments and ongoing costs.
Action: TODO
Action Input: Create a detailed financial plan including startup costs, operating expenses, and revenue projections for a vegan food truck.
Observation: 

1. Research the vegan food truck industry and gather information on startup costs, operating expenses, and revenue projections.
2. Create a budget spreadsheet to track all financial aspects of the food truck business.
3. Determine the initial startup costs, including purchasing a truck, equipment, permits, and licenses.
4. Research and compare prices for necessary equipment and supplies, such as a commercial kitchen, cooking appliances, and ingredients.
5. Estimate monthly operating expenses, including rent, utilities, insurance, and employee wages.
6. Develop a pricing strategy for menu items based on ingredient costs and market demand.
7. Create a sales forecast based on projected customer

{'objective': 'Write a business plan for a vegan food truck'}